In [5]:
from typing import Any

def extrac_contenido(ruta: str) -> list[str]:
    mi_lista = []
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = f.read()       
        lineas = contenido.split("\n")
        for linea in lineas:
            # Ignora el título y la cabecera
            if "registro de ventas" in linea.lower() or "id, producto" in linea.lower():
                continue

            if not linea.strip():
                continue

            if "instrucciones sugeridas" in linea.lower():
                break

            mi_lista.append(linea)
    return mi_lista


# contenido = extrac_contenido("../data/ventas_polos_aqp.txt")

In [24]:
class Venta:
    def __init__(
            self,
            id_venta: str,
            producto: str,
            talla: str,
            color: str,
            cliente: str,
            ciudad: str,
            cantidad: int,
            precio_unitario: float,
            metodo_pago: str,
            estado: str
    ) -> None:
        self.id_venta = id_venta
        self.producto = producto
        self.talla = talla
        self.color = color
        self.cliente = cliente
        self.ciudad = ciudad
        self.cantidad = cantidad
        self.precio_unitario = precio_unitario
        self.metodo_pago = metodo_pago
        self.estado = estado
        self.total_vendido = self.cantidad * self.precio_unitario

class RegistroVentas:
    def __init__(
            self,
            lineas_ventas: list[str]
    ) -> None:
        self.ventas: list[Venta] = []

        for linea in lineas_ventas:
            info = [item.strip() for item in linea.split(",")]  
            if len(info) < 10:
                continue
            venta = Venta(
                id_venta = info[0],
                producto = info[1],
                talla = info[2],
                color = info[3],
                cliente = info[4],
                ciudad = info[5],
                cantidad = int(info[6]),
                precio_unitario = float(info[7]),
                metodo_pago = info[8],
                estado = info[9]
            )    
            self.ventas.append(venta)


    def __len__(self) -> int:
        return len(self.ventas)

    def mostrar_estados_ventas(self) -> None:
        entregadas = sum(1 for v in self.ventas if v.estado.lower() == "entregado")
        pendientes = sum(1 for v in self.ventas if v.estado.lower() == "pendiente")
        canceladas = sum(1 for v in self.ventas if v.estado.lower() == "cancelado")
        print(f"Entregadas: {entregadas} | Pendientes: {pendientes} | Canceladas: {canceladas}")

    def ingreso_total_entregadas(self) -> float:
        return sum(v.total_vendido for v in self.ventas if v.estado.lower() == "entregado")

    def producto_mas_vendido(self) -> str:
        conteo = {}
        for v in self.ventas:
            conteo[v.producto] = conteo.get(v.producto, 0) + v.cantidad
        if not conteo:
            return "Ninguno"
        return max(conteo, key=conteo.get)

    def ventas_en_ciudad(self, ciudad: str) -> int:
        return sum(1 for v in self.ventas if v.ciudad.lower() == ciudad.lower())

    def metodo_pago_mas_usado(self) -> str:
        conteo = {}
        for v in self.ventas:
            conteo[v.metodo_pago] = conteo.get(v.metodo_pago, 0) + 1
        if not conteo:
            return "Ninguno"
        return max(conteo, key=conteo.get)

    def resumen_por_producto(self) -> dict[str, dict[str, Any]]:
        resumen = {}
        for v in self.ventas:
            if v.producto not in resumen:
                resumen[v.producto] = {"unidades_vendidas": 0, "ingreso_generado": 0.0}
            resumen[v.producto]["unidades_vendidas"] += v.cantidad
            resumen[v.producto]["ingreso_generado"] += v.total_vendido
        return resumen

    def export_ventas(self, ruta: str) -> None:
        with open(ruta, "w", encoding="utf-8") as f:
            f.write("id,producto,talla,color,cliente,ciudad,cantidad,precio_unitario,metodo_pago,estado,total_vendido\n")

        for v in self.ventas:
            with open(ruta, "a", encoding="utf-8") as f:
                f.write(f"{v.id_venta},{v.producto},{v.talla},{v.color},{v.cliente},{v.ciudad},{v.cantidad},{v.precio_unitario},{v.metodo_pago},{v.estado},{v.total_vendido}\n")


Instrucciones sugeridas para el ejercicio:
1. Lee el archivo línea por línea.
2. Ignora el título, la cabecera y las líneas vacías.
3. Separa cada venta usando split(",").
4. Calcula el total vendido por cada registro: cantidad * precio_unitario.
5. Muestra cuántas ventas fueron entregadas, pendientes y canceladas.
6. Calcula el ingreso total solo de las ventas entregadas.
7. Muestra cuál fue el producto más vendido por cantidad.
8. Muestra cuántas ventas se hicieron en Arequipa.
9. Muestra el método de pago más usado.
10. Reto extra: crea un resumen por producto con unidades vendidas e ingreso generado.


In [25]:

contenido = extrac_contenido("../data/ventas_polos_aqp.txt")
print(contenido)

registro = RegistroVentas(contenido)


# 5. Muestra cuántas ventas fueron entregadas, pendientes y canceladas.
registro.mostrar_estados_ventas()

# 6. Calcula el ingreso total solo de las ventas entregadas.
print(f"Total solo de ventas entregadas: S/. {registro.ingreso_total_entregadas()}")

# 7. Muestra cuál fue el producto más vendido por cantidad.
print(f"Producto más vendido por cantidad: {registro.producto_mas_vendido()}")

# 8. Muestra cuántas ventas se hicieron en Arequipa.
print(f"Número de ventas hechas en Arequipa: {registro.ventas_en_ciudad('Arequipa')}")

# 9. Muestra el método de pago más usado.
print(f"Método de pago más usado: {registro.metodo_pago_mas_usado()}")

# 10. Reto extra: crea un resumen por producto con unidades vendidas e ingreso generado.
for prod, datos in registro.resumen_por_producto().items():
    print(f"- {prod}: {datos['unidades_vendidas']} unidades, S/. {datos['ingreso_generado']}")


['V001, Polo Oversize Anime, M, Negro, Valeria Rojas, Arequipa, 2, 45.00, Yape, entregado', 'V002, Polo Streetwear Retro, L, Blanco, Diego Salazar, Arequipa, 1, 50.00, Efectivo, pendiente', 'V003, Polo Minimalista Logo, S, Gris, Camila Herrera, Cusco, 3, 38.00, Plin, entregado', 'V004, Polo Urbano 90s, M, Azul, Luis Ramos, Lima, 1, 55.00, Tarjeta, cancelado', 'V005, Polo Oversize Anime, XL, Negro, Andrea Torres, Arequipa, 2, 45.00, Yape, entregado', 'V006, Polo Gamer Edition, L, Rojo, Carlos Medina, Puno, 1, 48.00, Efectivo, pendiente', 'V007, Polo Streetwear Retro, M, Blanco, Sofía Vargas, Arequipa, 4, 50.00, Plin, entregado', 'V008, Polo Minimalista Logo, M, Verde, Jorge Flores, Tacna, 2, 38.00, Yape, entregado', 'V009, Polo Gamer Edition, S, Negro, María Quispe, Arequipa, 1, 48.00, Tarjeta, pendiente', 'V010, Polo Urbano 90s, L, Azul, Andrés Castillo, Cusco, 2, 55.00, Efectivo, entregado', 'V011, Polo Oversize Anime, M, Morado, Lucía Vargas, Arequipa, 1, 45.00, Yape, entregado', 'V0